## ML_1032: 2D Convolution

**Difficulty**: Medium | **TorchCode**: #22

### Problem
Implement a 2D convolution from scratch supporting stride and padding. Use `unfold` to extract patches and `einsum` to apply the kernel.

### Formula
$$O_{b,o,i,j} = \sum_{c,p,q} X_{b,c,\,si+p,\,sj+q} \cdot W_{o,c,p,q} + b_o$$
$$H_{\text{out}} = \left\lfloor\frac{H + 2p - k}{s}\right\rfloor + 1$$

In [ ]:
import torch
import torch.nn.functional as F

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    if padding > 0:
        x = F.pad(x, [padding] * 4)
    B, C_in, H, W = x.shape
    C_out, _, kH, kW = weight.shape
    H_out = (H - kH) // stride + 1
    W_out = (W - kW) // stride + 1
    patches = x.unfold(2, kH, stride).unfold(3, kW, stride)
    out = torch.einsum('bihwjk,oijk->bohw', patches, weight)
    if bias is not None:
        out = out + bias.view(1, -1, 1, 1)
    return out

In [ ]:
import torch
import torch.nn.functional as F

# --- Test 1: Output shape ---
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
assert my_conv2d(x, w).shape == (1, 16, 6, 6)

# --- Test 2: Matches F.conv2d ---
torch.manual_seed(0)
x = torch.randn(2, 3, 8, 8)
w = torch.randn(4, 3, 3, 3)
b = torch.randn(4)
out = my_conv2d(x, w, b)
ref = F.conv2d(x, w, b)
assert torch.allclose(out, ref, atol=1e-4)

# --- Test 3: With padding ---
torch.manual_seed(0)
x = torch.randn(1, 1, 5, 5)
w = torch.randn(1, 1, 3, 3)
out = my_conv2d(x, w, padding=1)
ref = F.conv2d(x, w, padding=1)
assert out.shape == ref.shape and torch.allclose(out, ref, atol=1e-4)

# --- Test 4: With stride ---
torch.manual_seed(0)
x = torch.randn(1, 1, 8, 8)
w = torch.randn(1, 1, 3, 3)
out = my_conv2d(x, w, stride=2)
ref = F.conv2d(x, w, stride=2)
assert out.shape == ref.shape and torch.allclose(out, ref, atol=1e-4)

# --- Test 5: Gradient flow ---
x = torch.randn(1, 1, 4, 4, requires_grad=True)
w = torch.randn(2, 1, 3, 3, requires_grad=True)
my_conv2d(x, w).sum().backward()
assert x.grad is not None and w.grad is not None

print('All tests passed!')